# Day 071 | RNN — Fundamentals

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning

**Author:** Ripon Chandra Malo

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Sequential data | Data where order matters | Settlement readings over months |
| RNN cell | Neural unit with memory of previous step | Each day's reading depends on prior history |
| Hidden state | Internal memory passed between time steps | Accumulated knowledge of ground behavior |
| `nn.RNN` | PyTorch RNN module | Building our sequence model |
| Vanishing gradient | Gradients shrink through long sequences | Why RNN struggles with long monitoring records |

---

## Our Geotechnical Problem

A construction site has **ground settlement monitoring sensors** that record vertical displacement daily.
Settlement is a **sequential process** — today's reading depends on yesterday's, which depends on the day before.

We will:
1. Understand why feedforward networks fail on sequential data
2. Learn how RNN cells maintain memory through hidden states
3. Build a simple RNN to predict the next settlement reading
4. Observe the vanishing gradient problem firsthand

---

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Why Sequential Data Needs Special Treatment

Consider ground settlement monitoring data. A sensor records vertical displacement every day:

```
Day 1:  -2.1 mm    (load just applied)
Day 2:  -4.5 mm    (primary consolidation begins)
Day 3:  -6.2 mm    (rate is slowing)
Day 4:  -7.4 mm    (approaching equilibrium)
Day 5:  ???         (what do we predict?)
```

The **order** matters. If we shuffle the data, we lose critical information about the consolidation trend.
A standard feedforward network sees each input independently. An RNN sees the **sequence**.

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE SYNTHETIC SETTLEMENT MONITORING DATA
# ───────────────────────────────────────────────────────
# Settlement follows a consolidation curve: S(t) = S_final * (1 - exp(-t/tau))
# We add noise to simulate real sensor readings.

n_days = 200
t = np.arange(1, n_days + 1, dtype=np.float32)

S_final = 50.0    # mm, ultimate settlement
tau = 40.0         # days, time constant (related to cv)
noise_std = 1.5    # mm, sensor noise

settlement = S_final * (1 - np.exp(-t / tau)) + np.random.normal(0, noise_std, n_days)
settlement = settlement.astype(np.float32)

plt.figure(figsize=(10, 4))
plt.plot(t, settlement, 'b-', alpha=0.7, label='Measured settlement')
plt.plot(t, S_final * (1 - np.exp(-t / tau)), 'r--', label='Theoretical curve')
plt.xlabel('Day')
plt.ylabel('Settlement (mm)')
plt.title('Ground Settlement Monitoring Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Data shape: {settlement.shape}")
print(f"Settlement range: {settlement.min():.1f} to {settlement.max():.1f} mm")

## Part 2: The RNN Cell — How Memory Works

An RNN cell processes one time step at a time, but carries a **hidden state** from the previous step:

```
h_t = tanh(W_ih * x_t  +  W_hh * h_{t-1}  +  bias)
```

Where:
- `x_t` = input at time step t (today's settlement)
- `h_{t-1}` = hidden state from previous step (memory of past readings)
- `h_t` = new hidden state (updated memory)
- `W_ih` = input-to-hidden weights
- `W_hh` = hidden-to-hidden weights (this is the recurrence!)

The hidden state is like an **engineer's mental model** of the site — it accumulates knowledge from all previous observations.

---

In [ ]:
# ───────────────────────────────────────────────────────
# MANUAL RNN CELL: Step through the math
# ───────────────────────────────────────────────────────
# Let's manually compute what happens inside an RNN cell
# to build intuition before using nn.RNN.

input_size = 1   # one feature: settlement reading
hidden_size = 4  # 4-dimensional hidden state

# Random weights (normally these are learned)
W_ih = np.random.randn(hidden_size, input_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b = np.zeros(hidden_size)

# Process first 5 settlement readings manually
h = np.zeros(hidden_size)  # initial hidden state is zeros

print("Step-by-step RNN processing:")
print("=" * 60)

for step in range(5):
    x_t = settlement[step:step+1]  # shape (1,)
    
    # RNN equation: h_t = tanh(W_ih @ x_t + W_hh @ h_{t-1} + b)
    h_new = np.tanh(W_ih @ x_t + W_hh @ h + b)
    
    print(f"\nDay {step+1}: Settlement = {x_t[0]:.2f} mm")
    print(f"  Hidden state (memory): [{', '.join(f'{v:.4f}' for v in h_new)}]")
    
    h = h_new  # carry memory forward

print("\nNotice: the hidden state changes each step, incorporating new input")
print("while retaining information from previous steps.")

## Part 3: PyTorch nn.RNN — The Real Deal

PyTorch provides `nn.RNN` which handles all the matrix operations efficiently.

Key parameters:
- `input_size`: number of features per time step
- `hidden_size`: dimension of hidden state (our choice — more = more capacity)
- `num_layers`: stack multiple RNN layers
- `batch_first=True`: input shape is (batch, seq_len, features)

---

In [ ]:
# ───────────────────────────────────────────────────────
# EXPLORE nn.RNN: Understand input/output shapes
# ───────────────────────────────────────────────────────

rnn = nn.RNN(input_size=1, hidden_size=16, num_layers=1, batch_first=True)

# Create a sample batch: 2 sequences, each 10 time steps, 1 feature
sample_input = torch.randn(2, 10, 1)  # (batch, seq_len, input_size)

# Forward pass
output, h_n = rnn(sample_input)

print("nn.RNN Shapes:")
print(f"  Input:          {list(sample_input.shape)}  (batch=2, seq_len=10, features=1)")
print(f"  Output:         {list(output.shape)}  (batch=2, seq_len=10, hidden=16)")
print(f"  Final hidden:   {list(h_n.shape)}  (layers=1, batch=2, hidden=16)")
print()
print("Output contains hidden states at ALL time steps.")
print("h_n contains the hidden state at the LAST time step only.")
print(f"\nVerify: output[:, -1, :] == h_n[0]: {torch.allclose(output[:, -1, :], h_n[0], atol=1e-6)}")

## Part 4: Preparing Sequences for the RNN

To train an RNN for next-step prediction, we create **sliding windows**:

```
Input sequence (window=5):  [s1, s2, s3, s4, s5]  --> Target: s6
                            [s2, s3, s4, s5, s6]  --> Target: s7
                            [s3, s4, s5, s6, s7]  --> Target: s8
```

The RNN reads 5 past readings and predicts the next one.

---

In [ ]:
# ───────────────────────────────────────────────────────
# CREATE SLIDING WINDOW SEQUENCES
# ───────────────────────────────────────────────────────

# Normalize settlement data to [0, 1]
s_min, s_max = settlement.min(), settlement.max()
s_norm = (settlement - s_min) / (s_max - s_min)

def create_sequences(data, window_size):
    """Create input-output pairs from time series using sliding window."""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

window_size = 10
X, y = create_sequences(s_norm, window_size)

# Reshape for RNN: (samples, seq_len, features)
X = X.reshape(-1, window_size, 1)
y = y.reshape(-1, 1)

# Train/test split (keep temporal order!)
split = int(0.8 * len(X))
X_train, X_test = torch.FloatTensor(X[:split]), torch.FloatTensor(X[split:])
y_train, y_test = torch.FloatTensor(y[:split]), torch.FloatTensor(y[split:])

print(f"Window size: {window_size} days")
print(f"Total sequences: {len(X)}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"X_train shape: {list(X_train.shape)} (samples, seq_len, features)")
print(f"y_train shape: {list(y_train.shape)} (samples, 1)")

## Part 5: Building the Settlement Prediction RNN

Our model architecture:
1. **RNN layer**: processes the sequence, outputs hidden states
2. **Linear layer**: maps final hidden state to settlement prediction

```
[s1, s2, ..., s10] --> RNN --> h_final --> Linear --> predicted s11
```

---

In [ ]:
# ───────────────────────────────────────────────────────
# SETTLEMENT PREDICTION RNN MODEL
# ───────────────────────────────────────────────────────

class SettlementRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        # RNN output: all hidden states; h_n: final hidden state
        out, h_n = self.rnn(x, h0)
        
        # Use the last hidden state for prediction
        last_hidden = out[:, -1, :]  # shape: (batch, hidden_size)
        prediction = self.fc(last_hidden)
        return prediction

model = SettlementRNN(input_size=1, hidden_size=32, num_layers=1)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN THE RNN
# ───────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

epochs = 100
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  Loss: {avg_loss:.6f}")

print("\nTraining complete.")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING LOSS CURVE
# ───────────────────────────────────────────────────────

plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'b-')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('RNN Training Loss for Settlement Prediction')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# EVALUATE ON TEST SET
# ───────────────────────────────────────────────────────

model.eval()
with torch.no_grad():
    pred_train = model(X_train).numpy()
    pred_test = model(X_test).numpy()

# Denormalize
pred_train_mm = pred_train * (s_max - s_min) + s_min
pred_test_mm = pred_test * (s_max - s_min) + s_min
y_train_mm = y_train.numpy() * (s_max - s_min) + s_min
y_test_mm = y_test.numpy() * (s_max - s_min) + s_min

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y_train_mm, pred_train_mm, 'bo', alpha=0.4, markersize=3)
axes[0].plot([y_train_mm.min(), y_train_mm.max()], 
             [y_train_mm.min(), y_train_mm.max()], 'r--')
axes[0].set_xlabel('Actual Settlement (mm)')
axes[0].set_ylabel('Predicted Settlement (mm)')
axes[0].set_title('Train: Predicted vs Actual')
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_test_mm, pred_test_mm, 'go', alpha=0.5, markersize=4)
axes[1].plot([y_test_mm.min(), y_test_mm.max()], 
             [y_test_mm.min(), y_test_mm.max()], 'r--')
axes[1].set_xlabel('Actual Settlement (mm)')
axes[1].set_ylabel('Predicted Settlement (mm)')
axes[1].set_title('Test: Predicted vs Actual')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
rmse = np.sqrt(mean_squared_error(y_test_mm, pred_test_mm))
mae = mean_absolute_error(y_test_mm, pred_test_mm)
r2 = r2_score(y_test_mm, pred_test_mm)
print(f"Test RMSE: {rmse:.3f} mm")
print(f"Test MAE:  {mae:.3f} mm")
print(f"Test R2:   {r2:.4f}")

## Part 6: Visualizing the Hidden State Evolution

Let's see how the RNN's hidden state (memory) evolves as it reads through a settlement time series.
This is analogous to an engineer building a mental picture of the consolidation process as they review daily readings.

---

In [ ]:
# ───────────────────────────────────────────────────────
# HIDDEN STATE EVOLUTION
# ───────────────────────────────────────────────────────

# Feed one long sequence through the RNN and capture all hidden states
long_seq = torch.FloatTensor(s_norm[:100]).reshape(1, 100, 1)

model.eval()
with torch.no_grad():
    output, _ = model.rnn(long_seq)
    hidden_states = output[0].numpy()  # (100, hidden_size)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Settlement data
axes[0].plot(settlement[:100], 'b-')
axes[0].set_ylabel('Settlement (mm)')
axes[0].set_title('Input: Settlement Time Series')
axes[0].grid(True, alpha=0.3)

# Hidden state dimensions (show first 4)
for i in range(4):
    axes[1].plot(hidden_states[:, i], label=f'h[{i}]', alpha=0.8)
axes[1].set_xlabel('Time Step (day)')
axes[1].set_ylabel('Hidden State Value')
axes[1].set_title('RNN Hidden State Evolution (first 4 dimensions)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Each hidden dimension captures a different aspect of the settlement pattern.")
print("Some track the trend, others the rate of change.")

## Part 7: The Vanishing Gradient Problem

The biggest limitation of vanilla RNNs: **vanishing gradients**.

During backpropagation through time (BPTT), gradients are multiplied by the recurrent weight matrix at each time step.
If eigenvalues of W_hh are < 1, the gradient **shrinks exponentially** with sequence length.

**In geotechnical terms:** An RNN trying to use Day 1 data to predict Day 100 would effectively "forget" the early readings because the gradient signal becomes vanishingly small.

This is why we need LSTM and GRU (Day 072).

---

In [ ]:
# ───────────────────────────────────────────────────────
# DEMONSTRATE VANISHING GRADIENTS
# ───────────────────────────────────────────────────────
# We compute the gradient of the loss w.r.t. the input at each time step.
# For a vanilla RNN, the gradient magnitude drops sharply for early steps.

# Create a long sequence
seq_len = 50
test_seq = torch.FloatTensor(s_norm[:seq_len]).reshape(1, seq_len, 1)
test_seq.requires_grad_(True)

# Forward pass
model.train()
target = torch.FloatTensor([[s_norm[seq_len]]])
pred = model(test_seq)
loss = criterion(pred, target)
loss.backward()

# Gradient magnitude at each time step
grad_magnitudes = test_seq.grad[0, :, 0].abs().detach().numpy()

plt.figure(figsize=(10, 4))
plt.bar(range(seq_len), grad_magnitudes, color='steelblue', alpha=0.7)
plt.xlabel('Time Step (days back from prediction)')
plt.ylabel('|Gradient|')
plt.title('Vanishing Gradient: Gradient Magnitude at Each Time Step')
plt.annotate('Recent inputs have\nstrong gradient signal', 
             xy=(seq_len-5, grad_magnitudes[-5]), fontsize=10,
             xytext=(seq_len-15, grad_magnitudes.max()*0.8),
             arrowprops=dict(arrowstyle='->'))
plt.annotate('Early inputs have\ntiny gradients (forgotten!)', 
             xy=(2, grad_magnitudes[2]), fontsize=10,
             xytext=(10, grad_magnitudes.max()*0.5),
             arrowprops=dict(arrowstyle='->'))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ratio = grad_magnitudes[-1] / (grad_magnitudes[0] + 1e-10)
print(f"Gradient at last step / first step: {ratio:.0f}x")
print("The RNN heavily favors recent readings and nearly ignores early ones.")
print("This is why vanilla RNNs struggle with long-term dependencies.")

---

## Practice Exercises

### Exercise 1: Change the Hidden Size
Modify the RNN to use `hidden_size=8` vs `hidden_size=64`. How does this affect prediction accuracy?

### Exercise 2: Stacked RNN
Set `num_layers=2` in the model. Does a deeper RNN improve performance on this data?

### Exercise 3: Different Window Sizes
Try `window_size` of 5, 20, and 50. Which window captures the consolidation trend best?

---

In [ ]:
# ───────────────────────────────────────────────────────
# EXERCISE SPACE
# ───────────────────────────────────────────────────────

# Exercise 1: Modify hidden_size
# model_small = SettlementRNN(hidden_size=8)
# model_large = SettlementRNN(hidden_size=64)
# Train both and compare...

# Exercise 2: Stacked RNN
# model_deep = SettlementRNN(hidden_size=32, num_layers=2)

# Exercise 3: Different window sizes
# for ws in [5, 20, 50]:
#     X_ws, y_ws = create_sequences(s_norm, ws)
#     ...

---

## Key Takeaways

1. **RNNs process sequential data** by maintaining a hidden state that carries information across time steps, making them ideal for time-series geotechnical monitoring data.

2. **The hidden state** acts as the network's memory — like an engineer who accumulates understanding of site behavior from sequential observations.

3. **Sliding windows** convert raw time series into input-target pairs for supervised training of sequence models.

4. **Vanishing gradients** cause vanilla RNNs to effectively forget early inputs in long sequences — recent data dominates predictions.

5. The vanishing gradient problem motivates **LSTM and GRU** (tomorrow), which add gating mechanisms to control what the network remembers and forgets.

---

## How This Connects to ML

| What We Did Today | What It Becomes in ML |
|-------------------|-----------------------|
| RNN for settlement prediction | Time series forecasting in monitoring systems |
| Hidden state as memory | Feature learning from sequential patterns |
| Sliding window sequences | Standard input preparation for all sequence models |
| Vanishing gradient demo | Motivation for LSTM/GRU architecture (Day 072) |
| Single-feature RNN | Foundation for multi-sensor RNNs (Day 073) |

---

*Day 071 of 100 — Now our networks have memory!*